# Multi-label MI Localization Dataset and DataLoaders

In [2]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import wfdb

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("ptbxl_localization_dataset")

RANDOM_SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 0  # Safer default for Windows/Jupyter
PIN_MEMORY = torch.cuda.is_available()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Device:", DEVICE)


Device: cuda


## Step 1: Resolve project and dataset paths

In [3]:
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = next(
    (
        parent
        for parent in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (parent / "data" / "raw" / "ptb-xl").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root containing data/raw/ptb-xl."
    )

DATASET_DIR = PROJECT_ROOT / "data" / "raw" / "ptb-xl"
LABELS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "localization"
    / "ptbxl_mi_localization_labels.csv"
)

if not LABELS_PATH.is_file():
    raise FileNotFoundError(
        f"{LABELS_PATH} not found. Run Notebook 08 successfully first."
    )

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATASET_DIR)
print("Localization labels:", LABELS_PATH)


Project root: c:\Users\sintu\Desktop\Myocardial Infarction
Dataset directory: c:\Users\sintu\Desktop\Myocardial Infarction\data\raw\ptb-xl
Localization labels: c:\Users\sintu\Desktop\Myocardial Infarction\data\processed\localization\ptbxl_mi_localization_labels.csv


## Step 2: Load labels and validate schema

In [4]:
labels = pd.read_csv(LABELS_PATH)

TARGET_COLUMNS = [
    "loc_inferior",
    "loc_anterior",
    "loc_anteroseptal",
    "loc_anterolateral",
    "loc_lateral",
    "loc_posterior",
    "loc_inferoposterior",
]

required_columns = {
    "ecg_id",
    "patient_id",
    "dataset_split",
    "has_mi_scp",
    "localization_mapping_complete",
    "filename_hr",
    *TARGET_COLUMNS,
}
missing = required_columns.difference(labels.columns)
if missing:
    raise ValueError(
        f"Localization label file is missing columns: {sorted(missing)}. "
        "Check Notebook 08 output."
    )

labels["dataset_split"] = labels["dataset_split"].astype(str).str.lower().str.strip()
labels["has_mi_scp"] = labels["has_mi_scp"].astype(bool)
labels["localization_mapping_complete"] = (
    labels["localization_mapping_complete"].astype(bool)
)

for column in TARGET_COLUMNS:
    labels[column] = pd.to_numeric(labels[column], errors="raise").astype(np.float32)
    invalid_values = set(labels[column].dropna().unique()) - {0.0, 1.0}
    if invalid_values:
        raise ValueError(f"{column} contains non-binary values: {invalid_values}")

display(labels.head())
print("All label rows:", len(labels))


,ecg_id,patient_id,dataset_split,filename_hr,has_mi_scp,localization_mapping_complete,mi_territories,unmapped_mi_codes,loc_inferior,loc_anterior,loc_anteroseptal,loc_anterolateral,loc_lateral,loc_posterior,loc_inferoposterior
0,1,15709.0,train,records500/00000/00001_hr,False,True,[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,13243.0,train,records500/00000/00002_hr,False,True,[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,20372.0,train,records500/00000/00003_hr,False,True,[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,17014.0,train,records500/00000/00004_hr,False,True,[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,17448.0,train,records500/00000/00005_hr,False,True,[],[],0.0,0.0,0.0,0.0,0.0,0.0,0.0


All label rows: 21799


## Step 3: Select eligible localization records

In [5]:
eligible = labels.loc[
    labels["has_mi_scp"]
    & labels["localization_mapping_complete"]
].copy()

eligible["dataset_split"] = eligible["dataset_split"].replace({"val": "validation"})
eligible = eligible.loc[
    eligible["dataset_split"].isin(["train", "validation", "test"])
].copy()

if eligible.empty:
    raise ValueError(
        "No eligible localization records found. Inspect Notebook 08 label audit."
    )

if eligible["filename_hr"].isna().any():
    examples = eligible.loc[eligible["filename_hr"].isna(), "ecg_id"].head().tolist()
    raise ValueError(f"Missing filename_hr for eligible ECG IDs: {examples}")

eligible["filename_hr"] = eligible["filename_hr"].astype(str)
eligible = eligible.drop_duplicates(subset=["ecg_id"])

print("Eligible localization records:", len(eligible))
display(
    eligible.groupby("dataset_split").agg(
        recordings=("ecg_id", "count"),
        patients=("patient_id", "nunique"),
    )
)


Eligible localization records: 4953


,recordings,patients
dataset_split,,
test,500,431
train,3969,3388
validation,484,437


## Step 4: Validate patient-level split separation and class coverage

In [6]:
patient_split_counts = eligible.groupby("patient_id")["dataset_split"].nunique()
leaking_patients = patient_split_counts[patient_split_counts > 1]

if not leaking_patients.empty:
    raise ValueError(
        "Patient leakage detected across splits. Examples: "
        f"{leaking_patients.index[:10].tolist()}"
    )

class_coverage = (
    eligible.groupby("dataset_split")[TARGET_COLUMNS]
    .sum()
    .astype(int)
)
display(class_coverage)

zero_classes = {
    split: class_coverage.loc[split][class_coverage.loc[split] == 0].index.tolist()
    for split in class_coverage.index
}
print("Territories with zero positives by split:", zero_classes)


,loc_inferior,loc_anterior,loc_anteroseptal,loc_anterolateral,loc_lateral,loc_posterior,loc_inferoposterior
dataset_split,,,,,,,
test,315,31,225,27,53,10,0
train,2529,271,1773,231,417,76,0
validation,313,28,228,29,51,11,0


Territories with zero positives by split: {'test': ['loc_inferoposterior'], 'train': ['loc_inferoposterior'], 'validation': ['loc_inferoposterior']}


## Step 5: ECG signal loading and per-record normalization

In [7]:
EXPECTED_SAMPLES = 5000
EXPECTED_LEADS = 12
SAMPLING_RATE = 500

def load_ecg_record(record_relative_path: str) -> np.ndarray:
    """Load one PTB-XL WFDB record and return normalized (12, 5000) data."""
    record_path = DATASET_DIR / record_relative_path
    record_path = record_path.with_suffix("") if record_path.suffix else record_path

    if not record_path.with_suffix(".hea").is_file():
        raise FileNotFoundError(
            f"WFDB header not found for record: {record_path.with_suffix('.hea')}"
        )

    signal, fields = wfdb.rdsamp(str(record_path))
    signal = np.asarray(signal, dtype=np.float32)

    if signal.shape != (EXPECTED_SAMPLES, EXPECTED_LEADS):
        raise ValueError(
            f"Unexpected ECG shape {signal.shape} for {record_relative_path}; "
            f"expected {(EXPECTED_SAMPLES, EXPECTED_LEADS)}."
        )

    if int(fields.get("fs", -1)) != SAMPLING_RATE:
        raise ValueError(
            f"Expected {SAMPLING_RATE} Hz, got {fields.get('fs')} Hz "
            f"for {record_relative_path}."
        )

    # Convert (samples, leads) -> (leads, samples).
    signal = signal.T

    mean = signal.mean(axis=1, keepdims=True)
    std = signal.std(axis=1, keepdims=True)
    std = np.maximum(std, 1e-6)
    signal = (signal - mean) / std

    if not np.isfinite(signal).all():
        raise ValueError(f"Non-finite values found in {record_relative_path}")

    return signal.astype(np.float32)


## Step 6: Define the PyTorch Dataset

In [8]:
class PTBXLLocalizationDataset(Dataset):
    """Lazy-loading PTB-XL dataset for multi-label MI territory prediction."""

    def __init__(
        self,
        frame: pd.DataFrame,
        target_columns: list[str],
    ) -> None:
        self.frame = frame.reset_index(drop=True).copy()
        self.target_columns = target_columns

        required = {"filename_hr", *target_columns}
        missing_columns = required.difference(self.frame.columns)
        if missing_columns:
            raise ValueError(f"Dataset frame missing columns: {sorted(missing_columns)}")

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.frame.iloc[index]
        ecg = load_ecg_record(row["filename_hr"])
        target = row[self.target_columns].to_numpy(dtype=np.float32)

        ecg_tensor = torch.from_numpy(ecg)
        target_tensor = torch.from_numpy(target)

        return ecg_tensor, target_tensor


## Step 7: Create train, validation, and test DataLoaders

In [9]:
train_frame = eligible.loc[eligible["dataset_split"] == "train"].copy()
val_frame = eligible.loc[eligible["dataset_split"] == "validation"].copy()
test_frame = eligible.loc[eligible["dataset_split"] == "test"].copy()

for split_name, split_frame in [
    ("train", train_frame),
    ("validation", val_frame),
    ("test", test_frame),
]:
    if split_frame.empty:
        raise ValueError(f"The {split_name} split is empty.")

train_dataset = PTBXLLocalizationDataset(train_frame, TARGET_COLUMNS)
val_dataset = PTBXLLocalizationDataset(val_frame, TARGET_COLUMNS)
test_dataset = PTBXLLocalizationDataset(test_frame, TARGET_COLUMNS)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


Train: 3969
Validation: 484
Test: 500


## Step 8: Validate one batch

In [10]:
signals, targets = next(iter(train_loader))

print("Signal batch shape:", tuple(signals.shape))
print("Target batch shape:", tuple(targets.shape))
print("Signal dtype:", signals.dtype)
print("Target dtype:", targets.dtype)
print("Signal finite:", bool(torch.isfinite(signals).all()))
print("Target values:", torch.unique(targets).tolist())

assert signals.ndim == 3
assert signals.shape[1] == EXPECTED_LEADS
assert signals.shape[2] == EXPECTED_SAMPLES
assert targets.ndim == 2
assert targets.shape[1] == len(TARGET_COLUMNS)
assert torch.isfinite(signals).all()
assert set(torch.unique(targets).tolist()).issubset({0.0, 1.0})

print("Batch validation passed.")


Signal batch shape: (32, 12, 5000)
Target batch shape: (32, 7)
Signal dtype: torch.float32
Target dtype: torch.float32
Signal finite: True
Target values: [0.0, 1.0]
Batch validation passed.


## Step 9: Save the dataset manifest

In [11]:
manifest_columns = [
    "ecg_id",
    "patient_id",
    "dataset_split",
    "filename_hr",
    *TARGET_COLUMNS,
]
manifest_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "localization"
    / "localization_dataset_manifest.csv"
)
eligible[manifest_columns].to_csv(manifest_path, index=False)

config = {
    "sampling_rate_hz": SAMPLING_RATE,
    "expected_leads": EXPECTED_LEADS,
    "expected_samples": EXPECTED_SAMPLES,
    "batch_size": BATCH_SIZE,
    "normalization": "per-record, per-lead z-score",
    "target_columns": TARGET_COLUMNS,
    "included_records": int(len(eligible)),
    "split_counts": eligible["dataset_split"].value_counts().to_dict(),
}
config_path = manifest_path.with_name("localization_dataset_config.json")
config_path.write_text(__import__("json").dumps(config, indent=2), encoding="utf-8")

print("Saved manifest:", manifest_path)
print("Saved config:", config_path)


Saved manifest: c:\Users\sintu\Desktop\Myocardial Infarction\data\processed\localization\localization_dataset_manifest.csv
Saved config: c:\Users\sintu\Desktop\Myocardial Infarction\data\processed\localization\localization_dataset_config.json
